# Smart Email Classification — Intent Model V3

This notebook trains and evaluates the V3 intent dataset.

V3 uses a **group-aware train/test split** so different variations of the same semantic scenario cannot appear in both training and testing.

Models tested:
1. TF-IDF + Multinomial Naive Bayes
2. TF-IDF + Linear SVM
3. TF-IDF + SelectKBest + SVM
4. Word TF-IDF + Character TF-IDF + Linear SVM
5. SVM and Naive Bayes hyperparameter tuning

We select the final model using **Macro F1 and accuracy**, rather than trying to force a target score.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import joblib

from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

sys.path.append("../")
from src.preprocessing import preprocess_text

## 2. Load V3 dataset

In [ ]:
df = pd.read_csv("../data/intent_emails_5000_v3.csv")

print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

## 3. Dataset validation

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate email texts:", df["text"].duplicated().sum())

print("\nIntent distribution:")
print(df["label"].value_counts().sort_index())

print("\nUnique semantic groups:", df["group_id"].nunique())
print("Emails per group:", df.groupby("group_id").size().unique())

## 4. Visualize class distribution

In [ ]:
plt.figure(figsize=(12, 6))

sns.countplot(
    data=df,
    x="label",
    order=df["label"].value_counts().index
)

plt.title("V3 Intent Distribution")
plt.xlabel("Intent")
plt.ylabel("Number of Emails")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Combine subject and email body

In [ ]:
df["combined_text"] = (
    "subject " + df["subject"].fillna("") +
    " body " + df["text"].fillna("")
)

df["clean_text"] = df["combined_text"].apply(preprocess_text)

print(df[["subject", "text", "clean_text", "label"]].head(5).to_string(index=False))

In [ ]:
print("Empty cleaned emails:",
      (df["clean_text"].str.strip() == "").sum())

print("Average cleaned length:",
      df["clean_text"].str.len().mean())

## 6. Group-aware train/test split

Every `group_id` contains 10 wording variations of one semantic scenario.

`StratifiedGroupKFold` ensures that a complete group stays in either training or testing.

We also verify that there is zero group overlap.

In [ ]:
X = df["clean_text"]
y = df["label"]
groups = df["group_id"]

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_groups = groups.iloc[train_idx]
test_groups = groups.iloc[test_idx]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training groups:", train_groups.nunique())
print("Testing groups:", test_groups.nunique())

overlap = set(train_groups) & set(test_groups)
print("Group overlap:", len(overlap))

assert len(overlap) == 0

In [ ]:
print("Training distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting distribution:")
print(y_test.value_counts().sort_index())

## 7. Model 1 — TF-IDF + Naive Bayes

In [ ]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.98,
        sublinear_tf=True
    )),
    ("classifier", MultinomialNB(alpha=0.5))
])

nb_pipeline.fit(X_train, y_train)
nb_predictions = nb_pipeline.predict(X_test)

nb_accuracy = accuracy_score(y_test, nb_predictions)
nb_f1 = f1_score(y_test, nb_predictions, average="macro")

print("Naive Bayes Accuracy:", nb_accuracy)
print("Naive Bayes Macro F1:", nb_f1)
print()
print(classification_report(y_test, nb_predictions))

## 8. Model 2 — TF-IDF + Linear SVM

In [ ]:
svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.98,
        sublinear_tf=True
    )),
    ("classifier", LinearSVC(
        C=0.5,
        class_weight="balanced",
        random_state=42
    ))
])

svm_pipeline.fit(X_train, y_train)
svm_predictions = svm_pipeline.predict(X_test)

svm_accuracy = accuracy_score(y_test, svm_predictions)
svm_f1 = f1_score(y_test, svm_predictions, average="macro")

print("SVM Accuracy:", svm_accuracy)
print("SVM Macro F1:", svm_f1)
print()
print(classification_report(y_test, svm_predictions))

## 9. Model 3 — Feature Selection + SVM

In [ ]:
feature_values = [1000, 2000, 3000, 5000, 8000, 10000]
feature_results = []

for k in feature_values:
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.98,
            sublinear_tf=True
        )),
        ("selector", SelectKBest(chi2, k=k)),
        ("classifier", LinearSVC(
            C=0.5,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    feature_results.append({
        "Features": k,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(y_test, predictions, average="macro")
    })

feature_results_df = pd.DataFrame(feature_results)
feature_results_df

## 10. Model 4 — Word + Character TF-IDF + Linear SVM

In [ ]:
word_char_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.98,
            max_features=10000,
            sublinear_tf=True
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=10000,
            sublinear_tf=True
        ))
    ])),
    ("classifier", LinearSVC(
        C=0.5,
        class_weight="balanced",
        random_state=42
    ))
])

word_char_pipeline.fit(X_train, y_train)
word_char_predictions = word_char_pipeline.predict(X_test)

word_char_accuracy = accuracy_score(y_test, word_char_predictions)
word_char_f1 = f1_score(y_test, word_char_predictions, average="macro")

print("Word + Character SVM Accuracy:", word_char_accuracy)
print("Word + Character SVM Macro F1:", word_char_f1)
print()
print(classification_report(y_test, word_char_predictions))

## 11. SVM C-value tuning

In [ ]:
c_values = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0]
c_results = []

for c in c_values:
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.98,
            sublinear_tf=True
        )),
        ("classifier", LinearSVC(
            C=c,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    c_results.append({
        "C": c,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(y_test, predictions, average="macro")
    })

c_results_df = pd.DataFrame(c_results)
c_results_df

## 12. Naive Bayes alpha tuning

In [ ]:
alpha_values = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
alpha_results = []

for alpha in alpha_values:
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=10000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=0.98,
            sublinear_tf=True
        )),
        ("classifier", MultinomialNB(alpha=alpha))
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    alpha_results.append({
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_test, predictions),
        "Macro F1": f1_score(y_test, predictions, average="macro")
    })

alpha_results_df = pd.DataFrame(alpha_results)
alpha_results_df

## 13. Compare the main models

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Naive Bayes",
        "Linear SVM",
        "Word + Character SVM"
    ],
    "Accuracy": [
        nb_accuracy,
        svm_accuracy,
        word_char_accuracy
    ],
    "Macro F1": [
        nb_f1,
        svm_f1,
        word_char_f1
    ]
}).sort_values("Macro F1", ascending=False).reset_index(drop=True)

comparison

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=comparison,
    x="Model",
    y="Accuracy"
)

plt.title("Intent Model Accuracy — V3")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 14. Feature-selection results

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    feature_results_df["Features"],
    feature_results_df["Accuracy"],
    marker="o"
)

plt.title("SVM Accuracy vs Selected Features")
plt.xlabel("Number of Selected Features")
plt.ylabel("Accuracy")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.show()

feature_results_df

## 15. Confusion matrix

In [ ]:
main_predictions = {
    "Naive Bayes": nb_predictions,
    "Linear SVM": svm_predictions,
    "Word + Character SVM": word_char_predictions
}

best_model_name = comparison.iloc[0]["Model"]
best_predictions = main_predictions[best_model_name]

print("Best main model:", best_model_name)

cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(12, 9))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=sorted(y.unique()),
    yticklabels=sorted(y.unique())
)

plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted Intent")
plt.ylabel("Actual Intent")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 16. Test unseen emails

In [ ]:
new_emails = [
    "Could you review the revised proposal before I send it to the client?",
    "My package has still not arrived and I need help checking the delivery.",
    "Please explain why the amount on my latest statement is different from what I expected.",
    "I wanted to follow up on my job application and ask about the next interview stage.",
    "Would Thursday afternoon work for our project discussion?",
    "Can you confirm my hotel reservation and arrival details for next week?",
    "Please let me know whether there is still space at the workshop.",
    "Are you free for dinner this weekend? It would be nice to catch up.",
    "The current member offer gives customers a discount on selected products.",
    "Here is the latest monthly update containing recent announcements and articles."
]

print("=" * 80)
print("NEW EMAIL PREDICTIONS")
print("=" * 80)

for email in new_emails:
    prediction = word_char_pipeline.predict([email])[0]

    print("\nEmail:")
    print(email)
    print("Predicted intent:", prediction)

## 17. Inspect SVM decision scores

In [ ]:
test_email = """
I am writing to ask about the interview process for the software developer
internship. Could you please tell me what the next step is?
"""

prediction = word_char_pipeline.predict([test_email])[0]
scores = word_char_pipeline.decision_function([test_email])[0]
classes = word_char_pipeline.classes_

score_table = pd.DataFrame({
    "Intent": classes,
    "Decision Score": scores
}).sort_values("Decision Score", ascending=False)

print("Predicted intent:", prediction)
score_table

## 18. Save V3 model pipelines

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(
    nb_pipeline,
    models_dir / "intent_naive_bayes_v3_pipeline.joblib"
)

joblib.dump(
    svm_pipeline,
    models_dir / "intent_svm_v3_pipeline.joblib"
)

joblib.dump(
    word_char_pipeline,
    models_dir / "intent_word_char_svm_v3_pipeline.joblib"
)

print("Models saved to:", models_dir.resolve())

## 19. Final interpretation

Use **Macro F1 together with accuracy** when selecting the model.

Do not artificially change the test data to obtain a target accuracy.

If V3 scores are lower than V2, that is useful evidence that the group-aware split is harder and reduces scenario/template leakage.

If Word + Character SVM performs best, we can use that pipeline as the production intent model. If normal SVM performs best, keep the simpler model.

The final model should be selected from these measured results, not from an expected accuracy number.